# 05 - Editing a package

Take objects out of a package into a new one, with everything they need; rename the HDF5
file; change an object and remove another; and judge the result. Every step works on copies:
Lutetia itself is never touched. It runs on **Lutetia**, a synthetic model (CC BY 4.0) built on boreholes of the BRGM Banque du sous-sol (Licence Ouverte 2.0), modified for the example: [geosiris-technologies/energyml-lutetia](https://github.com/geosiris-technologies/energyml-lutetia). The first run downloads it (27 MB) at the commit this repository pins; see [`_lutetia.py`](https://github.com/geosiris-technologies/energyml-rust/blob/main/examples/notebooks/_lutetia.py).

```
pip install energyml-rust matplotlib pandas
```

In [1]:
import tempfile
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

from energyml.rust import model
from energyml.rust.epc import Epc

from _lutetia import lutetia

root = lutetia()
source = Epc.open(str(root / "package" / "Lutetia.epc"))
out = Path(tempfile.mkdtemp())

## What an object needs

An object is rarely alone: a property needs its grid, the grid its interpretation, the
interpretation its feature, and so on. `needs` walks that closure, and says what is
referenced but not in the package - minus the objects the standard publishes in a catalogue,
whose absence means nothing.

In [2]:
porosity = source.find("ContinuousProperty", "Ypresian-Chalk grid - porosity")[0]
needed, missing = source.needs([porosity.uri])
print(f"{len(needed)} objects to take the porosity on its own; missing: {missing}")
pd.Series(Counter(u.split("(")[0].split(".")[-1] for u in needed)).sort_values(ascending=False).to_frame("objects")

123 objects to take the porosity on its own; missing: []


,objects
RockVolumeFeature,47
StratigraphicUnitInterpretation,47
StratigraphicColumnRankInterpretation,6
BoundaryFeature,5
FaultInterpretation,4
StructuralOrganizationInterpretation,2
Model,2
StratigraphicColumn,1
BoundaryFeatureInterpretation,1
LocalEngineering2dCrs,1


## Taking it out

`copy_from` copies the objects and their closure into another package, with their arrays.
The report says what was asked, what the closure added, and what could not travel.

In [3]:
extract = Epc.create("porosity.epc")
extract.set_data_root(str(out))
report = extract.copy_from(source, [porosity.uri], "copy")
print({key: len(value) if isinstance(value, list) else value for key, value in report.items()})
said = extract.save_as(str(out / "porosity.epc"))
print(*said, sep="\n")
sorted(p.name for p in out.iterdir())

{'objects': 123, 'added': 122, 'missing': 0, 'arrays': 2, 'arrays_missed': 0, 'unreadable': 0}
[warning] ContinuousProperty_bc5c4161-2f2a-50b7-8a97-1bb6db21265f.xml: `PropertyKind` references 69698278-5ad1-4512-bddc-8afed34bc3d8, which is not in this package; no relationship was written for it


['Lutetia.h5', 'porosity.epc']

The warning is right and harmless: the property's kind, *porosity*, is an object of the
Energistics catalogue. The package names it without carrying it, as the standard allows.

## Renaming the HDF5 file

The arrays went into a file still called `Lutetia.h5`. Renaming it touches the file and every
field and relationship that names it, so it is planned first. A plan that cannot be carried
out says why, and is refused whole rather than done in part.

In [4]:
extracted = Epc.open(str(out / "porosity.epc"))
print(extracted.plan_rename("porosity-data.h5", "x.h5")["refusals"])
plan = extracted.plan_rename("Lutetia.h5", "porosity.h5")
{key: plan[key] for key in ("allowed", "fields", "relationships", "refusals")}

['nothing in this package names `porosity-data.h5`']


{'allowed': True, 'fields': 2, 'relationships': 0, 'refusals': []}

In [5]:
done = extracted.rename_data_file("Lutetia.h5", "porosity.h5")
print({key: done[key] for key in ("fields", "relationships", "file")})
extracted.save_as(str(out / "porosity.epc"))
sorted(p.name for p in out.iterdir())

{'fields': 2, 'relationships': 0, 'file': 'moved'}


['porosity.epc', 'porosity.h5']

## Changing an object

An object is a dictionary: change it and `put` it back under the same uuid. Nothing is
written until `save_as`; `is_dirty` says whether there is something to write.

In [6]:
obj = extracted.get(porosity.uri)
obj["Citation"]["Title"] = "porosity, reviewed"
extracted.put(obj)
print("something to save:", extracted.is_dirty)

something to save: True


## Removing one, and what that breaks

The closure brought a collection of wells along. Before removing it, ask who points at it.

In [7]:
collection = extracted.find("DataobjectCollection")[0]
pd.DataFrame(extracted.referenced_by(collection.uri))[["qualified_type", "title", "path"]]

,qualified_type,title,path
0,resqml22.EarthModelInterpretation,Lutetia,WellboreInterpretationSet[0]


Remove it anyway, and the package is judged: the reference left behind is found, in the
`references` section, with the object and the field that hold it.

In [8]:
extracted.remove(collection.uri)
extracted.save_as(str(out / "porosity.epc"))
verdict = Epc.open(str(out / "porosity.epc")).check()
print({key: verdict[key] for key in ("errors", "warnings", "sound")})
[r["says"] for r in verdict["references"] if r["is_error"]]

{'errors': 1, 'warnings': 0, 'sound': False}


['WellboreInterpretationSet[0]: references 8fe8c4f1-a9d8-55c8-9e2c-ab4430aa244e (eml23.DataobjectCollection), which is not in this package']

The model says whether that field may simply go: `describe` lists it as not required. So
the repair is to drop it from the object that holds it.

In [9]:
holder = extracted.find("EarthModelInterpretation")[0]
field = next(f for f in model.describe("resqml22.EarthModelInterpretation")["fields"]
             if f["name"] == "WellboreInterpretationSet")
print("required:", field["required"])
obj = extracted.get(holder.uri)
del obj["WellboreInterpretationSet"]
extracted.put(obj)
extracted.save_as(str(out / "porosity.epc"))

required: False


['[warning] ContinuousProperty_bc5c4161-2f2a-50b7-8a97-1bb6db21265f.xml: `PropertyKind` references 69698278-5ad1-4512-bddc-8afed34bc3d8, which is not in this package; no relationship was written for it']

## Reading it back

Reopened as anyone would, the package says what was changed, still reads its array, and is
judged again.

In [10]:
again = Epc.open(str(out / "porosity.epc"))
values, shape = again.read_array(again.arrays(porosity.uri)[0])
print(again.get(porosity.uri)["Citation"]["Title"], shape, f"mean {np.nanmean(values):.3f}")
verdict = again.check()
{key: verdict[key] for key in ("errors", "warnings", "sound")}

porosity, reviewed [65520] mean 0.335


{'errors': 0, 'warnings': 0, 'sound': True}

## The other notebooks

- [00 - What is in this package?](https://github.com/geosiris-technologies/energyml-rust/blob/main/examples/notebooks/00-what-is-in-a-package.ipynb)
- [01 - Geometry, from 2D grids to 3D cells](https://github.com/geosiris-technologies/energyml-rust/blob/main/examples/notebooks/01-geometry-in-3d.ipynb)
- [02 - Arrays and properties](https://github.com/geosiris-technologies/energyml-rust/blob/main/examples/notebooks/02-arrays-and-properties.ipynb)
- [03 - Is this package valid?](https://github.com/geosiris-technologies/energyml-rust/blob/main/examples/notebooks/03-is-this-package-valid.ipynb)
- [04 - A package from nothing](https://github.com/geosiris-technologies/energyml-rust/blob/main/examples/notebooks/04-a-package-from-nothing.ipynb)
- [06 - Exporting to other tools](https://github.com/geosiris-technologies/energyml-rust/blob/main/examples/notebooks/06-exporting.ipynb)